# CSQA Natural-Error Activation Steering Analysis

Intervention notebook for steering naturally incorrect traces without synthetic corruption.

- model family: Qwen2.5
- target checkpoint: 3B instruct model
- intervention type: additive steering on decoder layer outputs
- steering methods:
  - contrastive mean direction
  - probe-normal direction
- decision space: constrained A-E answer-choice logits only


In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

from src.data.load_csqa import load_csqa

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)


## Configuration

Global study settings only.

- STEERING_LAYER_NUMBERS = None: all decoder layers are scanned
- STEERING_SCALES: additive steering strengths


In [ ]:
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
TRAIN_SPLIT = "train"
EVAL_SPLIT = "validation"
TRAIN_LIMIT = None
EVAL_LIMIT = None
STEERING_LAYER_NUMBERS = None
STEERING_SCALES = [0.5, 1.0, 2.0]
SEED = 42


## Environment Setup


In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)

LETTERS = ["A", "B", "C", "D", "E"]
MAX_SEQ_LEN = 384
EXTRACTION_BATCH_SIZE = 4
STEERING_BATCH_SIZE = 2

train_rows = load_csqa(split=TRAIN_SPLIT, limit=TRAIN_LIMIT).copy()
eval_rows = load_csqa(split=EVAL_SPLIT, limit=EVAL_LIMIT).copy()

for frame in [train_rows, eval_rows]:
    frame["n_choices"] = frame["csqa_choices"].map(len)
    frame["prompt_len_chars"] = frame["text"].str.len()

if torch.cuda.is_available():
    if torch.cuda.is_bf16_supported():
        model_dtype = torch.bfloat16
    else:
        model_dtype = torch.float16
    device_map = "auto"
else:
    model_dtype = torch.float32
    device_map = None

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=model_dtype,
    device_map=device_map,
    attn_implementation="eager",
)
model.eval()

input_device = model.device if hasattr(model, "device") else next(model.parameters()).device


def build_answer_token_ids(tok):
    out = {}
    for letter in LETTERS:
        ids = tok(" " + letter, add_special_tokens=False)["input_ids"]
        if len(ids) != 1:
            raise ValueError(f"Answer token '{letter}' is not single-token: {ids}")
        out[letter] = int(ids[0])
    return out


answer_token_ids = build_answer_token_ids(tok)
answer_ids = [answer_token_ids[l] for l in LETTERS]
answer_id_tensor = torch.tensor(answer_ids, dtype=torch.long)

display(eval_rows[["example_id", "answerKey", "prompt_len_chars"]].head())
print("train rows:", len(train_rows))
print("eval rows:", len(eval_rows))
print("answer token ids:", answer_token_ids)


## Helper Functions


In [ ]:
def get_decoder_layers(model):
    candidates = [
        "model.layers",
        "transformer.h",
        "gpt_neox.layers",
    ]
    for path in candidates:
        cur = model
        ok = True
        for part in path.split("."):
            if not hasattr(cur, part):
                ok = False
                break
            cur = getattr(cur, part)
        if ok:
            return cur
    raise ValueError("Could not locate decoder layers on this model.")


def encode_prompts(texts, tok, max_seq_len):
    batch = tok(
        list(texts),
        add_special_tokens=False,
        truncation=True,
        max_length=max_seq_len,
        padding=True,
        return_tensors="pt",
    )
    pos = []
    for mask in batch["attention_mask"]:
        nz = torch.nonzero(mask, as_tuple=False).view(-1)
        pos.append(int(nz[-1].item()))
    batch["decision_pos"] = torch.tensor(pos, dtype=torch.long)
    return batch


def unpack_output_hidden(output):
    if isinstance(output, tuple):
        return output[0]
    return output


def repack_output_hidden(output, new_hidden):
    if isinstance(output, tuple):
        return (new_hidden,) + tuple(output[1:])
    return new_hidden


def select_choice_logits(logits, decision_pos):
    row_idx = torch.arange(logits.shape[0], device=logits.device)
    answer_ids_on_device = answer_id_tensor.to(logits.device)
    return logits[row_idx, decision_pos][:, answer_ids_on_device].float()


def compute_choice_metrics(choice_logits, true_choice_idx):
    predicted_answer_index = choice_logits.argmax(dim=-1)
    row_idx = torch.arange(choice_logits.shape[0], device=choice_logits.device)
    true_logits = choice_logits[row_idx, true_choice_idx]
    other_logits = choice_logits.clone()
    other_logits[row_idx, true_choice_idx] = -torch.inf
    highest_other_logit = other_logits.max(dim=-1).values
    true_answer_vs_best_other_logit_gap = true_logits - highest_other_logit
    return predicted_answer_index.detach().cpu(), true_answer_vs_best_other_logit_gap.detach().cpu()


def apply_token_steering(hidden, decision_pos, direction, steering_scale):
    row_idx = torch.arange(hidden.shape[0], device=hidden.device)
    token_hidden = hidden[row_idx, decision_pos]
    rms = token_hidden.float().pow(2).mean(dim=-1, keepdim=True).sqrt().to(token_hidden.dtype)
    direction = direction.to(hidden.device, dtype=hidden.dtype)
    hidden_out = hidden.clone()
    hidden_out[row_idx, decision_pos] = token_hidden + (steering_scale * rms) * direction.unsqueeze(0)
    return hidden_out


decoder_layers = get_decoder_layers(model)
L = len(decoder_layers)

if STEERING_LAYER_NUMBERS is None:
    steering_layer_numbers = list(range(1, L + 1))
else:
    steering_layer_numbers = [int(x) for x in STEERING_LAYER_NUMBERS]

steering_layer_indices = [layer_number - 1 for layer_number in steering_layer_numbers]

print("decoder layers:", L)
print("steering layers:", steering_layer_numbers[:12], "..." if len(steering_layer_numbers) > 12 else "")


### Decision Score Used In All Summaries

	True_answer_vs_best_other_logit_gap means:

- logit of the true answer choice
- minus the highest logit among the remaining A-E choices

Positive change means the true answer became more competitive.


## Data Generation And Extraction

The train split is used to construct steering directions. The evaluation split is used to identify naturally incorrect traces and to test whether steering can convert them into correct outputs.


### Train Layer Extraction

Layer-output representations are extracted at the decision-position token for all steering layers.


In [ ]:
def extract_train_layer_outputs(frame, layer_numbers, batch_size, desc):
    rows = []
    hidden_by_layer = {layer_number: [] for layer_number in layer_numbers}

    for start in tqdm(range(0, len(frame), batch_size), total=int(math.ceil(len(frame) / batch_size)), desc=desc):
        batch_df = frame.iloc[start:start + batch_size].reset_index(drop=True)
        batch_cpu = encode_prompts(batch_df["text"], tok, MAX_SEQ_LEN)
        decision_pos = batch_cpu.pop("decision_pos")
        batch = {k: v.to(input_device) for k, v in batch_cpu.items()}
        decision_pos = decision_pos.to(input_device)
        true_choice_idx = torch.tensor([LETTERS.index(str(x)) for x in batch_df["answerKey"].tolist()], dtype=torch.long, device=input_device)

        with torch.inference_mode():
            out = model(**batch, return_dict=True, use_cache=False, output_hidden_states=True)

        choice_logits = select_choice_logits(out.logits, decision_pos)
        predicted_answer_index, true_answer_vs_best_other_logit_gap = compute_choice_metrics(choice_logits, true_choice_idx)
        clean_is_correct = predicted_answer_index.eq(true_choice_idx.detach().cpu())
        row_idx = torch.arange(len(batch_df), device=decision_pos.device)

        for layer_number in layer_numbers:
            hidden = out.hidden_states[layer_number][row_idx, decision_pos].detach().cpu().to(torch.float32)
            hidden_by_layer[layer_number].append(hidden)

        for i, row in batch_df.iterrows():
            rows.append(
                {
                    "example_id": row["example_id"],
                    "clean_prediction_idx": int(predicted_answer_index[i].item()),
                    "clean_true_answer_vs_best_other_logit_gap": float(true_answer_vs_best_other_logit_gap[i].item()),
                    "clean_is_correct": bool(clean_is_correct[i].item()),
                }
            )

    hidden_by_layer = {layer_number: torch.cat(blocks, dim=0) for layer_number, blocks in hidden_by_layer.items()}
    return pd.DataFrame(rows), hidden_by_layer


train_clean_df, train_hidden_by_layer = extract_train_layer_outputs(
    train_rows,
    steering_layer_numbers,
    EXTRACTION_BATCH_SIZE,
    desc="train layer extraction",
)

display(
    pd.DataFrame(
        [
            {
                "split": "train",
                "n_examples": len(train_clean_df),
                "clean_accuracy": float(train_clean_df["clean_is_correct"].mean()),
                "mean_true_answer_vs_best_other_logit_gap": float(train_clean_df["clean_true_answer_vs_best_other_logit_gap"].mean()),
            }
        ]
    ).round(4)
)


### Clean Evaluation Baseline

Naturally incorrect traces are defined by the clean evaluation run.


In [ ]:
def extract_clean_eval_metrics(frame, batch_size, desc):
    rows = []

    for start in tqdm(range(0, len(frame), batch_size), total=int(math.ceil(len(frame) / batch_size)), desc=desc):
        batch_df = frame.iloc[start:start + batch_size].reset_index(drop=True)
        batch_cpu = encode_prompts(batch_df["text"], tok, MAX_SEQ_LEN)
        decision_pos = batch_cpu.pop("decision_pos")
        batch = {k: v.to(input_device) for k, v in batch_cpu.items()}
        decision_pos = decision_pos.to(input_device)
        true_choice_idx = torch.tensor([LETTERS.index(str(x)) for x in batch_df["answerKey"].tolist()], dtype=torch.long, device=input_device)

        with torch.inference_mode():
            out = model(**batch, return_dict=True, use_cache=False)

        choice_logits = select_choice_logits(out.logits, decision_pos)
        predicted_answer_index, true_answer_vs_best_other_logit_gap = compute_choice_metrics(choice_logits, true_choice_idx)
        clean_is_correct = predicted_answer_index.eq(true_choice_idx.detach().cpu())

        for i, row in batch_df.iterrows():
            rows.append(
                {
                    "example_id": row["example_id"],
                    "clean_prediction_idx": int(predicted_answer_index[i].item()),
                    "clean_true_answer_vs_best_other_logit_gap": float(true_answer_vs_best_other_logit_gap[i].item()),
                    "clean_is_correct": bool(clean_is_correct[i].item()),
                }
            )

    return pd.DataFrame(rows)


eval_clean_df = extract_clean_eval_metrics(
    eval_rows,
    EXTRACTION_BATCH_SIZE,
    desc="eval clean baseline",
)

eval_bad_rows = (
    eval_rows.merge(
        eval_clean_df,
        on="example_id",
        how="left",
        validate="one_to_one",
    )
    .loc[lambda df: ~df["clean_is_correct"]]
    .reset_index(drop=True)
)

eval_baseline_summary = pd.DataFrame(
    [
        {
            "split": "validation",
            "n_examples": len(eval_clean_df),
            "clean_accuracy": float(eval_clean_df["clean_is_correct"].mean()),
            "mean_true_answer_vs_best_other_logit_gap": float(eval_clean_df["clean_true_answer_vs_best_other_logit_gap"].mean()),
            "n_natural_errors": int((~eval_clean_df["clean_is_correct"]).sum()),
        }
    ]
)

display(eval_baseline_summary.round(4))
display(eval_bad_rows[["example_id", "answerKey", "clean_prediction_idx", "clean_true_answer_vs_best_other_logit_gap"]].head())


### Steering Direction Construction

Two steering directions are constructed separately at each scanned layer:

- contrastive mean direction: mean(correct) - mean(incorrect)
- probe-normal direction: linear probe normal for clean correctness classification

The first direction is a direct class contrast. The second direction is the separating normal learned by a linear classifier at the same layer.


In [ ]:
probe_train_epochs = 100
probe_train_learning_rate = 5e-2
probe_train_weight_decay = 1e-4


def build_contrastive_mean_direction(hidden_cache, is_correct_mask):
    mask = torch.as_tensor(is_correct_mask.to_numpy() if hasattr(is_correct_mask, "to_numpy") else is_correct_mask, dtype=torch.bool)
    correct_mean = hidden_cache[mask].mean(dim=0)
    incorrect_mean = hidden_cache[~mask].mean(dim=0)
    raw_direction = (correct_mean - incorrect_mean).to(torch.float32)
    raw_norm = float(raw_direction.norm().item())
    direction = raw_direction / raw_direction.norm().clamp_min(1e-12)
    projection = hidden_cache @ direction

    return direction, {
        "steering_method": "Contrastive mean direction",
        "raw_direction_norm": raw_norm,
        "mean_projection_correct": float(projection[mask].mean().item()),
        "mean_projection_incorrect": float(projection[~mask].mean().item()),
        "probe_train_accuracy": np.nan,
    }


def build_probe_normal_direction(hidden_cache, is_correct_mask):
    mask = torch.as_tensor(is_correct_mask.to_numpy() if hasattr(is_correct_mask, "to_numpy") else is_correct_mask, dtype=torch.bool)
    x = hidden_cache.to(torch.float32)
    y = mask.to(torch.float32).unsqueeze(-1)
    mu = x.mean(dim=0)
    sigma = x.std(dim=0).clamp_min(1e-6)
    xz = (x - mu) / sigma

    probe = torch.nn.Linear(xz.shape[1], 1)
    optimizer = torch.optim.AdamW(
        probe.parameters(),
        lr=probe_train_learning_rate,
        weight_decay=probe_train_weight_decay,
    )

    for _ in tqdm(range(probe_train_epochs), desc="probe training"):
        optimizer.zero_grad(set_to_none=True)
        logits = probe(xz)
        loss = F.binary_cross_entropy_with_logits(logits, y)
        loss.backward()
        optimizer.step()

    with torch.inference_mode():
        logits = probe(xz)
        preds = logits.squeeze(-1).ge(0.0)
        train_accuracy = float(preds.eq(mask).float().mean().item())
        raw_weight = probe.weight.detach().cpu().squeeze(0).to(torch.float32)

    raw_direction = raw_weight / sigma.cpu()
    projection = x @ raw_direction
    if projection[mask].mean() < projection[~mask].mean():
        raw_direction = -raw_direction
        projection = -projection

    raw_norm = float(raw_direction.norm().item())
    direction = raw_direction / raw_direction.norm().clamp_min(1e-12)

    return direction, {
        "steering_method": "Probe-normal direction",
        "raw_direction_norm": raw_norm,
        "mean_projection_correct": float(projection[mask].mean().item()),
        "mean_projection_incorrect": float(projection[~mask].mean().item()),
        "probe_train_accuracy": train_accuracy,
    }


steering_directions = {}
direction_info_rows = []

for layer_number in tqdm(steering_layer_numbers, desc="steering direction construction"):
    hidden_cache = train_hidden_by_layer[layer_number]

    contrastive_direction, contrastive_info = build_contrastive_mean_direction(
        hidden_cache,
        train_clean_df["clean_is_correct"],
    )
    contrastive_info["steering_layer_number"] = layer_number
    contrastive_info["n_examples"] = int(hidden_cache.shape[0])
    contrastive_info["n_correct"] = int(train_clean_df["clean_is_correct"].sum())
    contrastive_info["n_incorrect"] = int((~train_clean_df["clean_is_correct"]).sum())
    steering_directions[(layer_number, "Contrastive mean direction")] = contrastive_direction
    direction_info_rows.append(contrastive_info)

    probe_direction, probe_info = build_probe_normal_direction(
        hidden_cache,
        train_clean_df["clean_is_correct"],
    )
    probe_info["steering_layer_number"] = layer_number
    probe_info["n_examples"] = int(hidden_cache.shape[0])
    probe_info["n_correct"] = int(train_clean_df["clean_is_correct"].sum())
    probe_info["n_incorrect"] = int((~train_clean_df["clean_is_correct"]).sum())
    steering_directions[(layer_number, "Probe-normal direction")] = probe_direction
    direction_info_rows.append(probe_info)

steering_direction_info_df = pd.DataFrame(direction_info_rows)
display(steering_direction_info_df.head(12).round(4))


### Steering Natural Bad Traces

Steering is applied only to evaluation examples that were incorrect in the clean run. The intervention target is the decoder layer output at the decision-position token.


In [ ]:
def run_natural_error_steering_scan(
    frame,
    steering_layer_numbers,
    steering_directions,
    steering_scales,
    batch_size,
):
    rows = []

    for steering_layer_number in tqdm(steering_layer_numbers, desc="steering scan"):
        steering_layer_index = steering_layer_number - 1
        steering_module = decoder_layers[steering_layer_index]

        for steering_method in ["Contrastive mean direction", "Probe-normal direction"]:
            direction = steering_directions[(steering_layer_number, steering_method)]

            for steering_scale in steering_scales:
                for start in range(0, len(frame), batch_size):
                    batch_df = frame.iloc[start:start + batch_size].reset_index(drop=True)
                    batch_cpu = encode_prompts(batch_df["text"], tok, MAX_SEQ_LEN)
                    decision_pos = batch_cpu.pop("decision_pos")
                    batch = {k: v.to(input_device) for k, v in batch_cpu.items()}
                    decision_pos = decision_pos.to(input_device)
                    true_choice_idx = torch.tensor([LETTERS.index(str(x)) for x in batch_df["answerKey"].tolist()], dtype=torch.long, device=input_device)

                    def steering_hook(module, inputs, output):
                        hidden = unpack_output_hidden(output)
                        hidden = apply_token_steering(hidden, decision_pos, direction, float(steering_scale))
                        return repack_output_hidden(output, hidden)

                    handle = steering_module.register_forward_hook(steering_hook)
                    try:
                        with torch.inference_mode():
                            out = model(**batch, return_dict=True, use_cache=False)
                    finally:
                        handle.remove()

                    choice_logits = select_choice_logits(out.logits, decision_pos)
                    predicted_answer_index, true_answer_vs_best_other_logit_gap = compute_choice_metrics(choice_logits, true_choice_idx)
                    steered_is_correct = predicted_answer_index.eq(true_choice_idx.detach().cpu())

                    for i, row in batch_df.iterrows():
                        rows.append(
                            {
                                "example_id": row["example_id"],
                                "steering_layer_number": steering_layer_number,
                                "steering_method": steering_method,
                                "steering_scale": float(steering_scale),
                                "steered_prediction_idx": int(predicted_answer_index[i].item()),
                                "steered_true_answer_vs_best_other_logit_gap": float(true_answer_vs_best_other_logit_gap[i].item()),
                                "steered_is_correct": bool(steered_is_correct[i].item()),
                            }
                        )

    return pd.DataFrame(rows)


steering_df = run_natural_error_steering_scan(
    eval_bad_rows,
    steering_layer_numbers,
    steering_directions,
    STEERING_SCALES,
    STEERING_BATCH_SIZE,
)

steering_df = steering_df.merge(
    eval_bad_rows[[
        "example_id",
        "clean_prediction_idx",
        "clean_true_answer_vs_best_other_logit_gap",
        "clean_is_correct",
    ]],
    on="example_id",
    how="left",
    validate="many_to_one",
)
steering_df["delta_true_answer_vs_best_other_logit_gap"] = (
    steering_df["steered_true_answer_vs_best_other_logit_gap"] - steering_df["clean_true_answer_vs_best_other_logit_gap"]
)
steering_df["prediction_changed"] = ~steering_df["steered_prediction_idx"].eq(steering_df["clean_prediction_idx"])
steering_df["became_correct"] = steering_df["steered_is_correct"]

display(steering_df.head())


## Basic Analysis

All rescue summaries below are computed on naturally incorrect evaluation examples only.


### Steering Summary

steering_scale is the multiplier applied to the normalized steering direction at the intervention layer.


In [ ]:
steering_summary = (
    steering_df.groupby(["steering_method", "steering_scale", "steering_layer_number"])
    .agg(
        mean_delta_true_answer_vs_best_other_logit_gap=("delta_true_answer_vs_best_other_logit_gap", "mean"),
        rescue_rate=("became_correct", "mean"),
        prediction_change_rate=("prediction_changed", "mean"),
        steered_mean_true_answer_vs_best_other_logit_gap=("steered_true_answer_vs_best_other_logit_gap", "mean"),
    )
    .reset_index()
)

steering_average_summary = (
    steering_summary.groupby(["steering_method", "steering_scale"])[
        ["mean_delta_true_answer_vs_best_other_logit_gap", "rescue_rate", "prediction_change_rate"]
    ]
    .mean()
    .reset_index()
)

display(steering_summary.round(4))
display(steering_average_summary.round(4))


In [ ]:
methods = steering_summary["steering_method"].drop_duplicates().tolist()
fig, axes = plt.subplots(len(methods), 2, figsize=(16, 5 * len(methods)), sharex=True)
if len(methods) == 1:
    axes = np.array([axes])

for row_idx, method in enumerate(methods):
    part = steering_summary.loc[steering_summary["steering_method"].eq(method)].sort_values(["steering_scale", "steering_layer_number"])
    for steering_scale in sorted(part["steering_scale"].unique()):
        scale_part = part.loc[part["steering_scale"].eq(steering_scale)].sort_values("steering_layer_number")
        x = scale_part["steering_layer_number"].to_numpy()

        axes[row_idx, 0].plot(
            x,
            scale_part["mean_delta_true_answer_vs_best_other_logit_gap"],
            marker="o",
            label=f"steering scale={steering_scale:g}",
        )
        axes[row_idx, 1].plot(
            x,
            scale_part["rescue_rate"],
            marker="o",
            label=f"steering scale={steering_scale:g}",
        )

    axes[row_idx, 0].axhline(0.0, color="black", linewidth=1, alpha=0.6)
    axes[row_idx, 0].set_ylabel("Mean delta")
    axes[row_idx, 0].set_title(f"{method}: Logit Gap Change")
    axes[row_idx, 0].legend()

    axes[row_idx, 1].set_ylabel("Dataset fraction")
    axes[row_idx, 1].set_title(f"{method}: Rescue Rate")
    axes[row_idx, 1].legend()

axes[-1, 0].set_xlabel("Steering layer")
axes[-1, 1].set_xlabel("Steering layer")
plt.tight_layout()
plt.show()


In [ ]:
display(
    steering_summary.sort_values(
        ["rescue_rate", "mean_delta_true_answer_vs_best_other_logit_gap"],
        ascending=[False, False],
    )
    .head(20)
    .round(4)
)
